# Week 75, Day 2: SQL II: joins and aggregation

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w075d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('shopdb.py', 'w', encoding='utf-8') as f:
    f.write('"""A small shop database for practising SQL (Week 75)."""\nimport sqlite3\n\nSCHEMA = """\nCREATE TABLE customers (\n    id     INTEGER PRIMARY KEY,\n    name   TEXT NOT NULL,\n    city   TEXT NOT NULL,\n    joined TEXT NOT NULL,\n    email  TEXT\n);\nCREATE TABLE products (\n    id       INTEGER PRIMARY KEY,\n    name     TEXT NOT NULL,\n    category TEXT NOT NULL,\n    price    REAL NOT NULL,\n    stock    INTEGER\n);\nCREATE TABLE orders (\n    id          INTEGER PRIMARY KEY,\n    customer_id INTEGER NOT NULL REFERENCES customers(id),\n    placed      TEXT NOT NULL,\n    status      TEXT NOT NULL\n);\nCREATE TABLE order_items (\n    order_id   INTEGER NOT NULL REFERENCES orders(id),\n    product_id INTEGER NOT NULL REFERENCES products(id),\n    qty        INTEGER NOT NULL,\n    unit_price REAL NOT NULL,\n    PRIMARY KEY (order_id, product_id)\n);\n"""\n\nDATA = """\nINSERT INTO customers VALUES\n    (1, \'Ada Lovelace\', \'London\', \'2024-01-15\', \'ada@example.com\'),\n    (2, \'Alan Turing\', \'Manchester\', \'2024-03-02\', \'alan@example.com\'),\n    (3, \'Grace Hopper\', \'New York\', \'2024-03-20\', NULL),\n    (4, \'Katherine Johnson\', \'Leeds\', \'2025-02-11\', \'kj@example.com\'),\n    (5, \'Tim Berners-Lee\', \'London\', \'2025-06-30\', NULL),\n    (6, \'Margaret Hamilton\', \'Bristol\', \'2025-09-09\', \'mh@example.com\');\nINSERT INTO products VALUES\n    (1, \'Tea\', \'drinks\', 3.50, 120),\n    (2, \'Coffee\', \'drinks\', 6.00, 80),\n    (3, \'Jam\', \'food\', 4.25, 0),\n    (4, \'Bread\', \'food\', 2.10, 35),\n    (5, \'Mug\', \'kitchen\', 8.00, 15),\n    (6, \'Teapot\', \'kitchen\', 24.00, 4),\n    (7, \'Biscuits\', \'food\', 1.80, 200),\n    (8, \'Hot chocolate\', \'drinks\', 5.50, NULL);\nINSERT INTO orders VALUES\n    (1, 1, \'2025-01-05\', \'delivered\'),\n    (2, 2, \'2025-01-17\', \'delivered\'),\n    (3, 1, \'2025-02-02\', \'cancelled\'),\n    (4, 3, \'2025-03-11\', \'delivered\'),\n    (5, 4, \'2025-03-29\', \'shipped\'),\n    (6, 1, \'2025-04-14\', \'delivered\'),\n    (7, 5, \'2025-05-01\', \'pending\'),\n    (8, 2, \'2025-05-20\', \'delivered\');\nINSERT INTO order_items VALUES\n    (1, 1, 2, 3.50), (1, 7, 3, 1.80),\n    (2, 2, 1, 6.00), (2, 5, 2, 8.00),\n    (3, 6, 1, 24.00),\n    (4, 1, 5, 3.25), (4, 3, 2, 4.25),\n    (5, 4, 1, 2.10), (5, 7, 2, 1.80), (5, 2, 1, 6.00),\n    (6, 6, 1, 22.00), (6, 5, 1, 8.00),\n    (7, 8, 2, 5.50),\n    (8, 1, 1, 3.50), (8, 4, 2, 2.10), (8, 7, 1, 1.80);\n"""\n\n\ndef connect():\n    """A fresh in-memory copy of the shop database."""\n    conn = sqlite3.connect(":memory:")\n    conn.executescript(SCHEMA + DATA)\n    return conn\n\n\ndef rows(sql, params=()):\n    """Run a query on a fresh copy of the database and return every row."""\n    conn = connect()\n    try:\n        return conn.execute(sql, params).fetchall()\n    finally:\n        conn.close()\n\n\ndef show(sql, params=()):\n    """Print a query\'s result as a small table."""\n    conn = connect()\n    cur = conn.execute(sql, params)\n    names = [d[0] for d in cur.description]\n    data = cur.fetchall()\n    widths = [max(len(str(x)) for x in [n] + [r[i] for r in data]) for i, n in enumerate(names)]\n    print("  ".join(n.ljust(w) for n, w in zip(names, widths)).rstrip())\n    for r in data:\n        print("  ".join(str(x).ljust(w) for x, w in zip(r, widths)).rstrip())\n    conn.close()\n')

**Predict before you run:** what will this print?



In [ ]:
from shopdb import show

show("""
SELECT c.name, o.id AS order_id, o.status
FROM orders AS o
JOIN customers AS c ON c.id = o.customer_id
WHERE c.city = 'London'
ORDER BY o.id
""")

In [ ]:
from shopdb import show

show("""
SELECT o.id, p.name, i.qty, i.unit_price, i.qty * i.unit_price AS line_total
FROM orders o
JOIN order_items i ON i.order_id = o.id
JOIN products p ON p.id = i.product_id
WHERE o.id IN (1, 4)
ORDER BY o.id, p.name
""")
print()
show("""
SELECT c.name, o.id AS order_id
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.id AND o.status = 'pending'
ORDER BY c.id
""")

In [ ]:
from shopdb import show

show("""
SELECT category, count(*) AS products, count(stock) AS known_stock,
       round(avg(price), 2) AS average, max(price) AS dearest
FROM products
GROUP BY category
ORDER BY category
""")
print()
show("""
SELECT o.id, round(sum(i.qty * i.unit_price), 2) AS total
FROM orders o JOIN order_items i ON i.order_id = o.id
WHERE o.status <> 'cancelled'
GROUP BY o.id
HAVING total > 20
ORDER BY total DESC
""")

In [ ]:
from shopdb import show

show("""
WITH lines AS (
    SELECT o.customer_id, i.product_id, i.qty
    FROM orders o JOIN order_items i ON i.order_id = o.id
    WHERE o.status <> 'cancelled'
)
SELECT c.name,
       (SELECT count(DISTINCT product_id) FROM lines l WHERE l.customer_id = c.id) AS different_products,
       (SELECT coalesce(sum(qty), 0) FROM lines l WHERE l.customer_id = c.id) AS items
FROM customers c
ORDER BY items DESC, c.name
""")

> **Key idea.** Normalised tables store each fact once, linked by foreign keys, with a table of its own for each many-to-many relationship; joins put facts back together. JOIN ... ON keeps matching pairs only; LEFT JOIN keeps every left row, with NULLs where nothing matched (find "none" with WHERE right.id IS NULL); conditions on the right table belong in ON, since in WHERE they undo the outer join. Aggregates (count, sum, avg, min, max) summarise rows, per group with GROUP BY; WHERE filters rows before grouping, HAVING filters groups after. Joins to many rows multiply rows (fan-out), so count DISTINCT or aggregate first. Subqueries (scalar, IN, EXISTS, correlated) and WITH build queries in steps; prefer NOT EXISTS to NOT IN with NULLs.

### Exercise: Joining two and three tables

Write two queries. `ORDER_FIVE`: the product name, quantity and unit price of every item in order 5, by product name. `MARCH`: the id, customer name and status of every order placed in March 2025, by order id.

In [ ]:
ORDER_FIVE = ___
MARCH = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_joins():\n    \"inner joins\"\n    from shopdb import rows\n    assert rows(ORDER_FIVE) == [(\"Biscuits\", 2, 1.8), (\"Bread\", 1, 2.1), (\"Coffee\", 1, 6.0)]\n    assert rows(MARCH) == [(4, \"Grace Hopper\", \"delivered\"), (5, \"Katherine Johnson\", \"shipped\")]")

### Exercise: The ones with nothing

Write two queries. `NO_ORDERS`: the names of customers who have never placed an order (use a `LEFT JOIN`). `NEVER_DELIVERED`: the names of products that have never been part of a **delivered** order (use `NOT EXISTS`), by name.

In [ ]:
NO_ORDERS = ___
NEVER_DELIVERED = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_outer():\n    \"finding what's missing\"\n    from shopdb import rows\n    assert rows(NO_ORDERS) == [(\"Margaret Hamilton\",)]\n    assert rows(NEVER_DELIVERED) == [(\"Hot chocolate\",)]\n    assert \"LEFT JOIN\" in NO_ORDERS.upper() and \"NOT EXISTS\" in NEVER_DELIVERED.upper()")

### Exercise: Groups and totals

Write three queries. `CATEGORIES`: each category with its number of products and average price (rounded to 2 places), by category. `TOTALS`: each order's id and total (`qty * unit_price` summed, rounded to 2 places), largest total first (ties by id). `BIG`: the ids of orders whose total is over 20, in id order (use `HAVING`).

In [ ]:
CATEGORIES = ___
TOTALS = ___
BIG = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_groups():\n    \"GROUP BY and HAVING\"\n    from shopdb import rows\n    assert rows(CATEGORIES) == [(\"drinks\", 3, 5.0), (\"food\", 3, 2.72), (\"kitchen\", 2, 16.0)]\n    assert rows(TOTALS) == [(6, 30.0), (4, 24.75), (3, 24.0), (2, 22.0), (1, 12.4), (5, 11.7), (7, 11.0), (8, 9.5)]\n    assert rows(BIG) == [(2,), (3,), (4,), (6,)] and \"HAVING\" in BIG.upper()")

### Exercise: What each customer spent

Write `SPEND`: for **every** customer (including those with no orders), their name, the number of their **delivered** orders, and the total they spent on delivered orders (rounded to 2 places, `0` if none), most spent first, ties by name. Watch out for fan-out: joining items multiplies the order rows.

In [ ]:
SPEND = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_spend():\n    \"outer joins, conditions in ON, and counting distinct\"\n    from shopdb import rows\n    assert rows(SPEND) == [(\"Ada Lovelace\", 2, 42.4), (\"Alan Turing\", 2, 31.5), (\"Grace Hopper\", 1, 24.75),\n                           (\"Katherine Johnson\", 0, 0), (\"Margaret Hamilton\", 0, 0), (\"Tim Berners-Lee\", 0, 0)]")

### Exercise: Queries inside queries

Write three queries. `ABOVE_AVERAGE`: the name and price of products dearer than the average price, dearest first (a scalar subquery). `LAST_ORDER`: each customer who has ordered, with the date of their latest order, latest first. `BEST_SELLERS`: the three products with the most units sold in orders that weren't cancelled, with the units, most first (ties by name).

In [ ]:
ABOVE_AVERAGE = ___
LAST_ORDER = ___
BEST_SELLERS = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_subqueries():\n    \"subqueries and grouped maximums\"\n    from shopdb import rows\n    assert rows(ABOVE_AVERAGE) == [(\"Teapot\", 24.0), (\"Mug\", 8.0)]\n    assert rows(LAST_ORDER) == [(\"Alan Turing\", \"2025-05-20\"), (\"Tim Berners-Lee\", \"2025-05-01\"), (\"Ada Lovelace\", \"2025-04-14\"),\n                                (\"Katherine Johnson\", \"2025-03-29\"), (\"Grace Hopper\", \"2025-03-11\")]\n    assert rows(BEST_SELLERS) == [(\"Tea\", 8), (\"Biscuits\", 6), (\"Bread\", 3)]")

### Exercise: Building a query in steps

Write `BIG_SPENDERS` using `WITH`: a first step computing each customer's total on delivered orders, then the names and totals (rounded to 2 places) of customers whose total is above the **average** of those totals, highest first.

In [ ]:
BIG_SPENDERS = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_cte():\n    \"a common table expression\"\n    from shopdb import rows\n    assert rows(BIG_SPENDERS) == [(\"Ada Lovelace\", 42.4)] and BIG_SPENDERS.strip().upper().startswith(\"WITH\")")

### Bug hunt: A customer with one order, who has none

`count(*)` counts rows, and the `LEFT JOIN` gives Margaret Hamilton one row (with `NULL`s for the order), so she's shown with 1 order: this prints `[('Ada Lovelace', 3), ('Alan Turing', 2), ('Grace Hopper', 1), ('Katherine Johnson', 1), ('Margaret Hamilton', 1), ('Tim Berners-Lee', 1)]`. Count only real orders, so she has 0.

In [ ]:
from shopdb import rows

print(rows("""
SELECT c.name, count(*) FROM customers c LEFT JOIN orders o ON o.customer_id = c.id
GROUP BY c.id ORDER BY c.name
"""))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"count a column, not rows\"\n    expect_output(\"[('Ada Lovelace', 3), ('Alan Turing', 2), ('Grace Hopper', 1), ('Katherine Johnson', 1), ('Margaret Hamilton', 0), ('Tim Berners-Lee', 1)]\")")

### Bug hunt: An outer join that went inner

The query should list **every** customer with their number of delivered orders, but the status test is in `WHERE`, which throws away the customers with no delivered orders (their status is `NULL`), so only three are printed. Move the test into the join's `ON`, so all six are printed.

In [ ]:
from shopdb import rows

for row in rows("""
SELECT c.name, count(o.id) FROM customers c
LEFT JOIN orders o ON o.customer_id = c.id
WHERE o.status = 'delivered'
GROUP BY c.id ORDER BY c.name
"""):
    print(row)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"every customer\"\n    lines = output.strip().splitlines()\n    assert len(lines) == 6 and lines[4] == \"('Margaret Hamilton', 0)\" and lines[0] == \"('Ada Lovelace', 2)\", output")

### Bug hunt: An aggregate in WHERE

`WHERE` runs before grouping, so it can't test `sum(...)`, and SQLite refuses the query with "misuse of aggregate". Filter the groups after aggregating instead, so this prints `[(2,), (3,), (4,), (6,)]`.

In [ ]:
from shopdb import rows

print(rows("SELECT order_id FROM order_items WHERE sum(qty * unit_price) > 20 GROUP BY order_id ORDER BY order_id"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"HAVING\"\n    expect_output(\"[(2,), (3,), (4,), (6,)]\")")

### Puzzle: A grouped report

Arrange the clauses of a query for the categories whose products have an average price over 4, with their averages. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # rows("SELECT count(*), count(email) FROM customers")[0]
q = ___   # rows("SELECT sum(price) FROM products WHERE price > 100")[0][0]
r = ___   # the number of rows from customers JOIN orders ON orders.customer_id = customers.id
s = ___   # rows("SELECT 1 NOT IN (2, NULL)")[0][0]
t = ___   # which runs first: "WHERE" or "HAVING"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == (6, 4) and q is None and r == 8, \"p, q and r\"\n    assert s is None and t == \"WHERE\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a monthly sales report

Write `REPORT`: one row per month (as `'YYYY-MM'`, from the order's date) that has **delivered** orders, in month order, with: the month, the number of delivered orders, the revenue from them (`qty * unit_price` summed, rounded to 2 places), and the name of that month's best-selling product by units in delivered orders (ties go to the product whose name comes first). Use `substr(placed, 1, 7)` for the month, and a correlated subquery (or a `WITH` step) for the best seller.

In [ ]:
REPORT = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_report():\n    \"grouping, fan-out and a correlated subquery\"\n    from shopdb import rows, connect\n    assert rows(REPORT) == [(\"2025-01\", 2, 34.4, \"Biscuits\"), (\"2025-03\", 1, 24.75, \"Tea\"), (\"2025-04\", 1, 30.0, \"Mug\"), (\"2025-05\", 1, 9.5, \"Bread\")]\n    conn = connect()\n    conn.execute(\"INSERT INTO orders VALUES (9, 6, '2025-05-28', 'delivered')\")\n    conn.execute(\"INSERT INTO order_items VALUES (9, 8, 4, 5.50)\")\n    assert conn.execute(REPORT).fetchall()[-1] == (\"2025-05\", 2, 31.5, \"Hot chocolate\")")

## Warm-up

`orders` only stores `customer_id`; the name is in `customers`. The **join** matches each order with the customer whose `id` equals its `customer_id`, giving rows that combine columns from both. Ada Lovelace appears three times because she placed three orders.

## Why tables are split

Why not one big table with the customer's name and city on every order? Because then each fact would be stored many times: change a customer's email, and you'd have to change it on every one of their orders, and miss some. Splitting data so that each fact is stored **once**, with **foreign keys** pointing between tables, is called **normalisation**:

- `customers` holds facts about customers; `products` facts about products.
- `orders` holds facts about an order (who placed it and when), pointing at its customer.
- `order_items` holds one row per product **in** an order: a **many-to-many** relationship (an order has many products, a product is in many orders) needs a table of its own. It records the `unit_price` actually paid, because a product's current price can change later.

The price of splitting is that most questions need **joins** to put the facts back together. That's what relational databases are built to do quickly.

## Joins

```sql
SELECT ...
FROM orders AS o
JOIN customers AS c ON c.id = o.customer_id
```

- **`JOIN ... ON condition`** (an **inner join**) pairs each row of the first table with each row of the second for which the condition is true. Rows with no match on the other side **disappear**.
- **Aliases** (`orders AS o`, or just `orders o`) keep queries short; qualify columns with them (`o.id`, `c.id`) whenever a name exists in both tables.
- Joins chain: `orders` to `order_items` to `products`, each with its own `ON`.
- **`LEFT JOIN`** keeps **every** row of the left table, filling the right table's columns with `NULL` where nothing matched. "Customers with no orders" is a `LEFT JOIN` plus `WHERE o.id IS NULL`.
- Where a condition goes matters with a `LEFT JOIN`: a condition in **`ON`** decides which right-hand rows match (unmatched left rows still appear), while the same condition in **`WHERE`** filters the joined result, throwing away the `NULL`-filled rows, and silently turning it back into an inner join.
- A table can be joined to **itself** (with two aliases), and `CROSS JOIN` gives every combination (rarely what you want).

## Aggregation

**Aggregate functions** turn many rows into one value: `count(*)` (rows), `count(column)` (rows where that column isn't `NULL`), `count(DISTINCT column)`, `sum`, `avg`, `min`, `max`, and in SQLite `group_concat(column, ', ')`. `sum` and `avg` ignore `NULL`s; `sum` of no rows is `NULL`, so wrap it in `coalesce(sum(x), 0)`.

- **`GROUP BY columns`** splits the rows into groups with equal values in those columns, and the aggregates are computed **per group**: one result row per group.
- Every column in the `SELECT` must either be in the `GROUP BY` or inside an aggregate. (SQLite is lenient here and picks a value from some row; other databases refuse the query. Don't rely on the leniency.)
- **`WHERE`** filters rows **before** grouping; **`HAVING`** filters **groups after** aggregating, so it can test aggregates: `HAVING sum(qty * unit_price) > 20`.

The full evaluation order: `FROM` and joins, `WHERE`, `GROUP BY`, aggregates, `HAVING`, `SELECT`, `DISTINCT`, `ORDER BY`, `LIMIT`.

**Fan-out.** Joining a table to one with several matching rows **multiplies** rows: a customer with two orders of three items each appears six times. Counting or summing after such a join counts the multiplied rows. Use `count(DISTINCT o.id)`, or aggregate each table before joining.

## Subqueries and WITH

A query can contain another:

- A **scalar subquery** in brackets gives one value: `WHERE price > (SELECT avg(price) FROM products)`.
- **`IN (SELECT ...)`** and **`EXISTS (SELECT ...)`** test membership: "products that appear in some delivered order". Beware `NOT IN` with a subquery that can return `NULL`: `x NOT IN (1, NULL)` is unknown, never true; `NOT EXISTS` is safer.
- A **correlated** subquery refers to the outer row: `(SELECT max(placed) FROM orders o WHERE o.customer_id = c.id)`, evaluated once per customer.
- A subquery in `FROM` is a temporary table: `FROM (SELECT ...) AS t`.

**`WITH`** (a **common table expression**, CTE) names a subquery up front, so a complex query reads as a series of steps:

```sql
WITH totals AS (
    SELECT o.customer_id, sum(i.qty * i.unit_price) AS spent
    FROM orders o JOIN order_items i ON i.order_id = o.id
    WHERE o.status = 'delivered'
    GROUP BY o.customer_id
)
SELECT c.name, t.spent
FROM totals t JOIN customers c ON c.id = t.customer_id
WHERE t.spent > (SELECT avg(spent) FROM totals);
```

`UNION` (removing duplicates) and `UNION ALL` (keeping them) stack the results of two queries with the same columns.

## Practice

As yesterday, write each query as a string; the checks run it on a fresh copy of the shop database.

## Quiz

## Daily challenge

## Before you go

Answer yesterday's question about the shop with a join or a group. Tomorrow you'll drive the database from Python properly with `sqlite3`: connections, parameters (the safe way to put values in queries), transactions and rows as dictionaries.